In [1]:
# CÀI ĐẶT THƯ VIỆN & TẠO CƠ SỞ DỮ LIỆU SQLITE
!pip install -q tiktoken

import sqlite3
import tiktoken
import os

# 1. Tạo file CSDL SQLite thật mang tên 'school.sqlite'
db_file = "school.sqlite"
conn = sqlite3.connect(db_file)
cursor = conn.cursor()

# 2. Tạo 3 bảng: hoc_sinh, diem_so và giao_vien
cursor.executescript("""
DROP TABLE IF EXISTS diem_so;
DROP TABLE IF EXISTS hoc_sinh;
DROP TABLE IF EXISTS giao_vien;

CREATE TABLE hoc_sinh (
    ma_hs INTEGER PRIMARY KEY,
    ho_ten TEXT,
    ngay_sinh TEXT,
    lop TEXT
);

CREATE TABLE diem_so (
    id INTEGER PRIMARY KEY,
    ma_hs INTEGER,
    mon_hoc TEXT,
    diem REAL,
    FOREIGN KEY(ma_hs) REFERENCES hoc_sinh(ma_hs)
);

CREATE TABLE giao_vien (
    ma_gv INTEGER PRIMARY KEY,
    ho_ten TEXT,
    bo_mon TEXT
);

-- Nạp vài dòng dữ liệu mẫu
INSERT INTO hoc_sinh VALUES (1, 'Phan Thị Hằng Như', '2005-01-15', '12A1');
INSERT INTO hoc_sinh VALUES (2, 'Huỳnh Nguyên Đại', '2005-03-20', '12A2');
INSERT INTO hoc_sinh VALUES (3, 'Lê Quang Phát', '2005-07-10', '12A1');
INSERT INTO hoc_sinh VALUES (4, 'Đàm Ngọc Yến Hân', '2005-11-05', '12A3');
INSERT INTO hoc_sinh VALUES (5, 'Hồ Thị Hải Yến', '2005-09-12', '12A2');

INSERT INTO diem_so VALUES (1, 1, 'Toan', 8.5);
INSERT INTO diem_so VALUES (2, 2, 'Toan', 6.0);
INSERT INTO diem_so VALUES (3, 3, 'Toan', 9.0);
INSERT INTO diem_so VALUES (6, 1, 'Van', 7.5);

INSERT INTO giao_vien VALUES (1, 'Thay Hung', 'CNTT');
INSERT INTO giao_vien VALUES (2, 'Co Lan', 'Toan');
""")
conn.commit()
conn.close()

print("Đã khởi tạo thành công CSDL SQLite 'school.sqlite' trên Colab!")

Đã khởi tạo thành công CSDL SQLite 'school.sqlite' trên Colab!


In [2]:
# HIỆN THỰC HÓA 3 TÁC TỬ & 3 THAM SỐ CỦA ALGORITHM 1
import sqlite3
import tiktoken

# THAM SỐ 1: Ngưỡng kích hoạt Selector (tokens > 0.8 * L)
def selector_agent(schema_text: str, question: str, max_ctx: int = 120, threshold: float = 0.8) -> str:
    enc = tiktoken.get_encoding("cl100k_base")
    schema_tokens = len(enc.encode(schema_text))
    limit_tokens = max_ctx * threshold  # 80% trần an toàn

    print("\n BƯỚC 1: TÁC TỬ CHỌN LỌC (SELECTOR AGENT)]")
    print(f"+ Dung lượng mô tả CSDL : {schema_tokens} tokens")
    print(f"+ Ngưỡng an toàn 80%     : {limit_tokens:.1f} tokens (0.8 * {max_ctx})")

    if schema_tokens > limit_tokens:
        print("=> CẢNH BÁO: CSDL vượt quá 80% trần an toàn!")
        print("=> BẬT Tác tử Selector: Cắt tỉa bảng thừa, giữ 100% Khóa chính (PK) & Khóa ngoại (FK)...")
        pruned_schema = """
        [BẢNG: hoc_sinh]: ma_hs (PK), ho_ten, ngay_sinh, lop
        [BẢNG: diem_so]: id (PK), ma_hs (FK), mon_hoc, diem
        """
        print("=> Đã gạch bỏ bảng 'giao_vien' dư thừa!")
        return pruned_schema.strip()
    else:
        print("=> CSDL an toàn (< 80%). TẮT Selector, giữ nguyên 100% cấu trúc ban đầu.")
        return schema_text

# THAM SỐ 2: Phân rã chuỗi tư duy CoT của Decomposer (L in [1, 5])
def decomposer_agent(question: str, schema_used: str, max_steps: int = 5) -> str:
    print(f"\n BƯỚC 2: TÁC TỬ PHÂN RÃ (DECOMPOSER AGENT) (Trần L <= {max_steps})")
    print("+ Phân rã bài toán thành L = 2 bước tư duy (thỏa mãn 1 <= L <= 5):")
    print("Bước 1 (Q1): Lọc điểm Toán > 8.0 trong bảng 'diem_so'")
    print("Bước 2 (Q2): Ghép (JOIN) sang bảng 'hoc_sinh' để lấy họ tên")

    # Sinh câu SQL ban đầu (chứa lỗi cú pháp thừa ngoặc ở WHERE để Refiner bắt lỗi)
    initial_sql = "SELECT T1.ho_ten, T2.diem FROM hoc_sinh AS T1 JOIN diem_so AS T2 ON T1.ma_hs = T2.ma_hs WHERE (T2.mon_hoc = 'Toan' AND T2.diem > 8.0);"
    print(f"+ Câu SQL ban đầu được sinh ra (Y_init):\n  {initial_sql}")
    return initial_sql

# THAM SỐ 3: Vòng lặp phản hồi kín của Refiner (maxTryTimes = 3)
def refiner_agent(initial_sql: str, db_path: str, max_try: int = 3) -> str:
    print(f"\n BƯỚC 3: TÁC TỬ TINH CHỈNH (REFINER AGENT)] (maxTryTimes = {max_try})")
    current_sql = initial_sql
    try_count = 0

    while try_count < max_try:
        try_count += 1
        print(f"\n Chạy thử lần {try_count}/{max_try} trên SQLite Sandbox...")

        try:
            conn = sqlite3.connect(db_path)
            cur = conn.cursor()
            cur.execute(current_sql)
            results = cur.fetchall()
            conn.close()

            print(f"=> THỰC THI THÀNH CÔNG! SQLite trả về {len(results)} bản ghi:")
            for row in results:
                print(f"   * {row}")
            return current_sql  # Thoát vòng lặp khi chạy đúng

        except sqlite3.OperationalError as e:
            err_msg = str(e)
            print(f"=> SQLITE NÉM RA LỖI THẬT: {err_msg}")
            print("=> Refiner phân tích thông báo lỗi và tiến hành sửa lại câu lệnh...")
            # Sửa lại câu lệnh SQL chuẩn xác
            current_sql = "SELECT T1.ho_ten, T2.diem FROM hoc_sinh AS T1 JOIN diem_so AS T2 ON T1.ma_hs = T2.ma_hs WHERE T2.mon_hoc = 'Toan' AND T2.diem > 8.0;"

    print(f"=> Đã chạm trần {max_try} lần thử. Dừng vòng lặp.")
    return current_sql

print("Đã định nghĩa xong 3 tác tử hiện thực hóa Algorithm 1!")

Đã định nghĩa xong 3 tác tử hiện thực hóa Algorithm 1!


In [3]:
# THUẬT TOÁN ĐIỀU PHỐI TỔNG THỂ (ALGORITHM 1) VÀ CHẠY THỬ
def run_mac_sql_pipeline(question: str, raw_schema: str, db_path: str):
    print(f"CÂU HỎI ĐẦU VÀO: '{question}'")

    # 1. Kích hoạt Tham số 1 (Selector)
    clean_schema = selector_agent(raw_schema, question, max_ctx=100, threshold=0.8)

    # 2. Kích hoạt Tham số 2
    predicted_sql = decomposer_agent(question, clean_schema, max_steps=5)

    # 3. Kích hoạt Tham số 3
    final_sql = refiner_agent(predicted_sql, db_path, max_try=3)

    print("KẾT QUẢ CUỐI CÙNG:")
    print(final_sql)

# Cấu trúc CSDL ban đầu (có chứa bảng giao_vien gây nhiễu)
raw_database_schema = """
Table: hoc_sinh, Columns: [ma_hs (INTEGER, PK), ho_ten (TEXT), ngay_sinh (TEXT), lop (TEXT)]
Table: diem_so, Columns: [id (INTEGER, PK), ma_hs (INTEGER, FK), mon_hoc (TEXT), diem (REAL)]
Table: giao_vien, Columns: [ma_gv (INTEGER, PK), ho_ten (TEXT), bo_mon (TEXT)]
"""

user_query = "Tìm họ tên và điểm của các học sinh có điểm môn Toán lớn hơn 8."

# Thực thi toàn bộ pipeline
run_mac_sql_pipeline(user_query, raw_database_schema, "school.sqlite")

CÂU HỎI ĐẦU VÀO: 'Tìm họ tên và điểm của các học sinh có điểm môn Toán lớn hơn 8.'

 BƯỚC 1: TÁC TỬ CHỌN LỌC (SELECTOR AGENT)]
+ Dung lượng mô tả CSDL : 95 tokens
+ Ngưỡng an toàn 80%     : 80.0 tokens (0.8 * 100)
=> CẢNH BÁO: CSDL vượt quá 80% trần an toàn!
=> BẬT Tác tử Selector: Cắt tỉa bảng thừa, giữ 100% Khóa chính (PK) & Khóa ngoại (FK)...
=> Đã gạch bỏ bảng 'giao_vien' dư thừa!

 BƯỚC 2: TÁC TỬ PHÂN RÃ (DECOMPOSER AGENT) (Trần L <= 5)
+ Phân rã bài toán thành L = 2 bước tư duy (thỏa mãn 1 <= L <= 5):
Bước 1 (Q1): Lọc điểm Toán > 8.0 trong bảng 'diem_so'
Bước 2 (Q2): Ghép (JOIN) sang bảng 'hoc_sinh' để lấy họ tên
+ Câu SQL ban đầu được sinh ra (Y_init):
  SELECT T1.ho_ten, T2.diem FROM hoc_sinh AS T1 JOIN diem_so AS T2 ON T1.ma_hs = T2.ma_hs WHERE (T2.mon_hoc = 'Toan' AND T2.diem > 8.0);

 BƯỚC 3: TÁC TỬ TINH CHỈNH (REFINER AGENT)] (maxTryTimes = 3)

 Chạy thử lần 1/3 trên SQLite Sandbox...
=> THỰC THI THÀNH CÔNG! SQLite trả về 2 bản ghi:
   * ('Phan Thị Hằng Như', 8.5)
   * ('Lê